# Matrix Smartboard: QLoRA Qwen3.5-4B (Kaggle T4)
Нужно: Accelerator = GPU T4, Internet = On, добавить датасет с train.jsonl/val.jsonl/test.jsonl.

In [ ]:
!pip install -q -U "transformers>=5.6" peft "bitsandbytes>=0.45" accelerate flash-linear-attention
import glob, os
print(glob.glob('/kaggle/input/**/*.jsonl', recursive=True))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import json, os, re, time, glob, inspect, torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = "unsloth/Qwen3.5-4B"
def find(name): return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]
TRAIN, VAL, TEST = find("train.jsonl"), find("val.jsonl"), find("test.jsonl")
OUT = "/kaggle/working/lora-qwen3.5-4b"
MAX_LEN = 1024
BF16 = torch.cuda.is_bf16_supported()
DT = torch.bfloat16 if BF16 else torch.float16
print("bf16:", BF16)

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
def encode(msgs):
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    full = prompt + msgs[-1]["content"] + "<|im_end|>\n"
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    f = tok(full, add_special_tokens=False)["input_ids"][:MAX_LEN]
    lab = [-100] * min(len(p), len(f)) + f[len(p):]
    return {"input_ids": f, "labels": lab[:len(f)]}
class DS(Dataset):
    def __init__(self, path):
        self.rows = [encode(json.loads(l)["messages"]) for l in open(path, encoding="utf-8")]
        self.rows = [r for r in self.rows if any(x != -100 for x in r["labels"])]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]
def collate(b):
    n = max(len(x["input_ids"]) for x in b)
    ids = torch.full((len(b), n), tok.pad_token_id); lab = torch.full((len(b), n), -100); att = torch.zeros((len(b), n), dtype=torch.long)
    for i, x in enumerate(b):
        L = len(x["input_ids"]); ids[i,:L] = torch.tensor(x["input_ids"]); lab[i,:L] = torch.tensor(x["labels"]); att[i,:L] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": att}
train, val = DS(TRAIN), DS(VAL)
print(len(train), len(val))

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DT, bnb_4bit_use_double_quant=True)
model = AutoModelForImageTextToText.from_pretrained(BASE, quantization_config=bnb, device_map={"": 0}, dtype=DT)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
targets = set()
for name, mod in model.named_modules():
    if isinstance(mod, torch.nn.Linear) or mod.__class__.__name__ == "Linear4bit":
        if re.search(r"(visual|vision|lm_head|mlp\.gate$|router|shared_expert_gate)", name): continue
        targets.add(name.split(".")[-1])
print(sorted(targets))
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=r"^(?!.*(visual|vision)).*\.(" + "|".join(sorted(targets)) + r")$"))
model.print_trainable_parameters()
kw = dict(output_dir=OUT, num_train_epochs=2, per_device_train_batch_size=1, gradient_accumulation_steps=16,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=10, bf16=BF16, fp16=not BF16, max_grad_norm=1.0,
    optim="paged_adamw_8bit", gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5, eval_strategy="steps", eval_steps=40, save_strategy="steps", save_steps=40, save_total_limit=2,
    per_device_eval_batch_size=1, report_to="none", remove_unused_columns=False, dataloader_num_workers=0)
allowed = set(inspect.signature(TrainingArguments.__init__).parameters)
tr = Trainer(model=model, args=TrainingArguments(**{k: v for k, v in kw.items() if k in allowed}),
             train_dataset=train, eval_dataset=val, data_collator=collate)
t0 = time.time(); tr.train()
print(f"train done in {(time.time()-t0)/60:.1f} min")
model.save_pretrained(OUT + "/final"); tok.save_pretrained(OUT + "/final")
json.dump(tr.state.log_history, open(OUT + "/log_history.json", "w"), indent=1)

In [ ]:
# test on 44 held-out questions
SYSTEM = ("Ты — помощник по алгебре и началам анализа для 10 класса (Казахстан). "
          "Отвечай на языке вопроса: на русском или на казахском. Решай по шагам, формулы пиши в LaTeX.")
model.eval(); model.config.use_cache = True
rows = [json.loads(l) for l in open(TEST, encoding="utf-8")]
with open("/kaggle/working/finetuned_4b.jsonl", "w", encoding="utf-8") as f:
    for i, r in enumerate(rows, 1):
        ids = tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": r["q"]}],
              add_generation_prompt=True, enable_thinking=False, return_tensors="pt", return_dict=True).to(0)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=900, do_sample=False, repetition_penalty=1.05)
        r["model_output"] = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        f.write(json.dumps(r, ensure_ascii=False) + "\n"); f.flush(); print(i, r["id"])
!cd /kaggle/working && zip -qr lora-qwen3.5-4b.zip lora-qwen3.5-4b/final lora-qwen3.5-4b/log_history.json && ls -la